# ML Feature Engineering — Base Features (Phase A), Split Manifests (Phase B) & Dev Audit

Olist Brazilian E-Commerce — IT5006 Group 11 · ML dataset pipeline.

This notebook **imports and calls** the reusable modules under `src/features/`
(per `data/business/ml/feature_contract.md` v1.2 §11.2 — it does **not** embed
copies of their source), reproduces every `data/business/ml/` artifact
end-to-end, and adds a **development-only feature audit** for the two eligible
task populations.

| Artifact | Phase |
| --- | --- |
| `orders_ml_features.csv` | A |
| `feature_schema.json` | A |
| `dataset_manifest.json` | A |
| `zip_centroids.csv` | A |
| `quality_report.json` | A |
| `split_assignments.csv` | B |
| `cv_assignments.csv` | B |
| `split_report.json` | B |

**Lineage:** `data/preprocessed/*.csv → src/features/ → data/business/ml/`.
ML never reads `data/raw/` or `data/business/dashboard/`.

## 0. Environment & repo root

Locate the repository root relative to wherever the notebook runs, so the
`src.features` package resolves without hard-coded paths.

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "preprocessed").is_dir() and (p / "src" / "features").is_dir():
            return p
    raise FileNotFoundError("repo root not found (needs data/preprocessed and src/features)")

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import numpy, pandas, sklearn, geopandas, shapely, pyproj
print("pandas", pandas.__version__, "| numpy", numpy.__version__)
print("sklearn", sklearn.__version__, "| geopandas", geopandas.__version__,
      "| shapely", shapely.__version__, "| pyproj", pyproj.__version__)

repo root: E:\NUS最后一学期\IT5006\小组作业数据集\IT5006_GRP11


pandas 2.2.2 | numpy 2.4.2
sklearn 1.9.0 | geopandas 1.2.0 | shapely 2.1.2 | pyproj 3.7.2


## 1. Input inventory

The only transactional inputs are `data/preprocessed/*.csv`. The single
external reference is the pinned IBGE `BR_Pais_2024` country polygon under
`docs/references/geography/` (a documented, label-independent quality-control
exception). Row counts below mirror `dataset_manifest.json`.

In [2]:
PRE = ROOT / "data" / "preprocessed"
ML = ROOT / "data" / "business" / "ml"

print("preprocessed inputs:")
for f in sorted(PRE.glob("*.csv")):
    with open(f, encoding="utf-8") as fh:
        n = sum(1 for _ in fh) - 1
    print(f"  {f.name:45s} {n:>8,} rows")

print()
print("boundary reference dir:", ROOT / "docs" / "references" / "geography")
print("(lineage: data/preprocessed -> src/features -> data/business/ml; "
       "no data/raw or dashboard read)")

preprocessed inputs:
  olist_customers_dataset.csv                     99,441 rows
  olist_geolocation_dataset.csv                  738,332 rows


  olist_order_items_dataset.csv                  112,650 rows
  olist_order_payments_dataset.csv               103,886 rows
  olist_order_reviews_dataset.csv                104,719 rows
  olist_orders_dataset.csv                        99,441 rows
  olist_products_dataset.csv                      32,951 rows
  olist_sellers_dataset.csv                        3,095 rows
  product_category_name_translation.csv               71 rows

boundary reference dir: E:\NUS最后一学期\IT5006\小组作业数据集\IT5006_GRP11\docs\references\geography
(lineage: data/preprocessed -> src/features -> data/business/ml; no data/raw or dashboard read)


## 2. Reusable implementation (`src/features`) — imported, not embedded

`feature_contract.md` §11.2 requires the notebook to **import reusable modules**,
not maintain embedded copies of their full implementation (embedded copies
drift from the executed source). The authoritative implementation lives in:

| Module | Responsibility |
| --- | --- |
| `src/features/contract.py` | Frozen 27-column predictor allowlist, valid UF codes, usable payment types, column roles, serialized `feature_schema`. |
| `src/features/geography.py` | IBGE boundary provenance/CRS checks, ZIP normalization, Haversine, boundary-inclusive polygon filtering, ZIP/state medians, distance/interstate features. |
| `src/features/aggregation.py` | Order-grain aggregation of items/payments/reviews (no item×payment×review Cartesian product). |
| `src/features/checks.py` | Deterministic synthetic fixture checks exercising the exact contract rules. |
| `src/features/serialization.py` | Canonical-LF writers (LF content hashes require `\n` line endings). |
| `src/features/build_features.py` | Base-table builder + §9 acceptance checks + quality report. Entrypoint: `python -m src.features.build_features`. |
| `src/features/create_splits.py` | Order-level `StratifiedGroupKFold` development/holdout + inner CV manifests. Entrypoint: `python -m src.features.create_splits`. |

The driver cells in sections 3–4 call these exact modules, so running this
notebook reproduces the same artifacts as the CLI entrypoints.

## 3. Phase A — build the base table

`python -m src.features.build_features` (exact CLI path). Loads the preprocessed
inputs, validates primary keys, prepares the IBGE boundary, builds the
polygon-filtered ZIP/state centroid lookup, aggregates items/payments/reviews to
the order grain, computes distance/interstate geography features, assembles the
one-row-per-order base table, exports CSV/schema/manifest and runs the §9
acceptance checks. It fails loudly on any required check failure.

In [3]:
from src.features import build_features as bf

# Exact equivalent of `python -m src.features.build_features` (build_features.main).
bf.main()

build_features complete.
  orders:      99441 rows -> E:\NUS最后一学期\IT5006\小组作业数据集\IT5006_GRP11\data\business\ml\orders_ml_features.csv
  centroids:   19023 keys -> E:\NUS最后一学期\IT5006\小组作业数据集\IT5006_GRP11\data\business\ml\zip_centroids.csv
  eligible_regression:    96470
  eligible_classification: 98673
  checks: 27/27 passed (see quality_report.json)


In [4]:
import json
import pandas as pd

base = pd.read_csv(ML / "orders_ml_features.csv", low_memory=False)
print("orders_ml_features.csv:", base.shape)
print("columns:", list(base.columns))
print()
print("base table preview:")
print(base[["order_id", "n_items", "total_price", "distance_km_max",
            "primary_payment_type", "lead_days", "is_detractor",
            "eligible_regression", "eligible_classification"]].head().to_string())

centroids = pd.read_csv(ML / "zip_centroids.csv")
print()
print("zip_centroids.csv:", centroids.shape)
print(centroids.head(3).to_string())

q = json.loads((ML / "quality_report.json").read_text(encoding="utf-8"))
print()
print("checks summary:", q["checks_summary"])
print("geography reference_filter:", json.dumps(q["geography"]["reference_filter"], indent=2))
print("geography keys:", json.dumps(q["geography"]["keys"], indent=2))
print()
print("cohort counts:")
for k, v in q["cohort_counts"].items():
    print(f"  {k}: {v}")
print()
print("predictor missingness (non-zero only):")
for k, v in q["predictor_missingness"].items():
    if v:
        print(f"  {k}: {v}")
print()
print("deferred checks:", q["deferred_checks"])

orders_ml_features.csv: (99441, 43)
columns: ['order_id', 'customer_id', 'customer_unique_id', 'prediction_timestamp', 'order_status', 'lead_days', 'review_score_min', 'is_detractor', 'eligible_regression', 'eligible_classification', 'regression_exclusion_reason', 'classification_exclusion_reason', 'regression_label_available_at', 'review_latest_observed_at', 'review_count', 'seller_id_missing', 'n_items', 'has_items', 'n_products', 'n_sellers', 'n_categories', 'total_price', 'total_freight', 'freight_ratio', 'freight_ratio_missing', 'total_weight_g', 'total_volume_cm3', 'weight_missing_fraction', 'volume_missing_fraction', 'primary_category', 'category_missing_fraction', 'customer_state', 'primary_seller_state', 'distance_km_max', 'distance_missing_fraction', 'interstate_share', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'primary_payment_type', 'payment_installments_max', 'n_payment_methods', 'payment_missing']

base table preview:
                           order_id  n_

## 4. Phase B — build the split manifests

`python -m src.features.create_splits` (exact CLI path). Reads the base table,
assigns each order to `development`/`holdout`/`excluded` via outer
`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` on resolved
customer groups with three **order-level** strata (negative / nonnegative /
unknown); fold 0 is the holdout. Inner task folds are independent five-fold
splits within development (StratifiedGroupKFold for classification, GroupKFold
for regression).

In [5]:
from src.features import create_splits as cs

# Exact equivalent of `python -m src.features.create_splits` (create_splits.main).
cs.main()

create_splits complete.
  split_assignments.csv: 99441 orders (holdout 19889, dev 79552, excluded 0)
  cv_assignments.csv: 156077 rows


In [6]:
import json
import pandas as pd

sr = json.loads((ML / "split_report.json").read_text(encoding="utf-8"))
print("split_version:", sr["split_version"], "| contract_version:", sr["contract_version"])
print("outer_config:", json.dumps(sr["outer_config"], indent=2))
print("counts:", json.dumps(sr["counts"], indent=2))
print("checks:")
for c in sr["checks"]:
    print(f"  [{c['status']:4s}] {c['id']} — {c['detail']}")

sa = pd.read_csv(ML / "split_assignments.csv")
print()
print("split_assignments.csv:", sa.shape)
print(sa["split_assignment"].value_counts().to_string())

cv = pd.read_csv(ML / "cv_assignments.csv")
print()
print("cv_assignments.csv:", cv.shape)
print(cv.groupby("task")["validation_fold"].value_counts().to_string())

split_version: v1.1 | contract_version: v1.2
outer_config: {
  "n_splits": 5,
  "shuffle": true,
  "random_state": 42,
  "holdout_fold": 0,
  "strata": [
    "negative",
    "nonnegative",
    "unknown"
  ],
  "strata_grain": "per order, grouped by customer_unique_id"
}
counts: {
  "holdout_orders": 19889,
  "development_orders": 79552,
  "excluded_orders": 0,
  "cv_classification_rows": 78938,
  "cv_regression_rows": 77139
}
checks:
  [pass] outer_groups_disjoint — holdout 19246 groups disjoint from development 76850 groups
  [pass] classification_both_classes_development — development classification classes [np.float64(0.0), np.float64(1.0)]
  [pass] classification_both_classes_each_fold — both classes present in every classification validation fold
  [pass] no_holdout_in_cv — no holdout order assigned to an inner validation fold

split_assignments.csv: (99441, 5)
split_assignment
development    79552
holdout        19889



cv_assignments.csv: (156077, 4)
task            validation_fold
classification  1                  15789
                0                  15788
                2                  15787
                3                  15787
                4                  15787
regression      0                  15428
                1                  15428
                2                  15428
                3                  15428
                4                  15427


## 5. Development-only feature audit (`feature_contract.md` §11.2)

The audit below is **development-only** (holdout rows are excluded from every
feature/target relationship and distribution), run **separately for the two
eligible task populations** (regression: `eligible_regression == 1`;
classification: `eligible_classification == 1`). It does **not** impute, scale,
clip, or drop rows, and it does **not** choose features or transformations — it
only reports roles/dtypes/units, missingness, numeric summaries, category
frequencies, distributions, correlations, and written observations.

Full-population integrity, eligibility and split-balance counts are reported
first (permissible); no holdout feature/target relationship is used anywhere.

In [7]:
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

base = pd.read_csv(ML / "orders_ml_features.csv", low_memory=False)
split = pd.read_csv(ML / "split_assignments.csv")
base = base.merge(split[["order_id", "split_assignment"]], on="order_id", how="left")

schema = json.loads((ML / "feature_schema.json").read_text(encoding="utf-8"))
ALLOW = schema["predictor_allowlist"]          # 27 frozen predictors
NUMS = schema["numeric_columns"]               # 20 numeric predictors
CATS = schema["categorical_columns"]           # 7 categorical predictors
ROLE = {c["name"]: (c["role"], c["dtype"], c.get("unit", "-"))
        for c in schema["columns"]}

print("=== Full-population integrity / eligibility / split balance ===")
print("orders:", len(base))
print(base["split_assignment"].value_counts(dropna=False).to_string())
print()
for col in ["eligible_regression", "eligible_classification"]:
    print(f"{col}: {int(base[col].sum())} eligible / {len(base)} orders")
print()
print("regression exclusion reasons:")
print(base["regression_exclusion_reason"].value_counts(dropna=False).to_string())
print()
print("classification exclusion reasons:")
print(base["classification_exclusion_reason"].value_counts(dropna=False).to_string())

dev = base[base["split_assignment"] == "development"].copy()
reg = dev[dev["eligible_regression"] == 1].copy()
cls = dev[dev["eligible_classification"] == 1].copy()
print()
print("=== Development-only populations (audit scope) ===")
print(f"development orders: {len(dev)}")
print(f"regression population:   {len(reg)}")
print(f"classification population: {len(cls)}")

=== Full-population integrity / eligibility / split balance ===
orders: 99441
split_assignment
development    79552
holdout        19889

eligible_regression: 96470 eligible / 99441 orders
eligible_classification: 98673 eligible / 99441 orders

regression exclusion reasons:
regression_exclusion_reason
NaN                           96470
not_delivered                  2963
missing_delivery_timestamp        8

classification exclusion reasons:
classification_exclusion_reason
NaN                      98673
missing_review_target      768

=== Development-only populations (audit scope) ===
development orders: 79552
regression population:   77139
classification population: 78938


In [8]:
# Column roles, dtypes and units (population-independent; from feature_schema.json)
roles = pd.DataFrame(
    [{"column": c, "role": ROLE[c][0], "dtype": ROLE[c][1], "unit": ROLE[c][2]} for c in ALLOW]
)
print("=== 27 predictor column roles / dtypes / units ===")
print(roles.to_string(index=False))
print()
print("targets / eligibility / identifiers:")
for c in ["order_id", "customer_unique_id", "lead_days", "is_detractor",
          "eligible_regression", "eligible_classification"]:
    r, d, u = ROLE[c]
    print(f"  {c:28s} role={r:12s} dtype={d}")

=== 27 predictor column roles / dtypes / units ===
                   column    role  dtype unit
                  n_items feature    int    -
                has_items feature    int    -
               n_products feature    int    -
                n_sellers feature    int    -
             n_categories feature    int    -
              total_price feature  float  BRL
            total_freight feature  float  BRL
            freight_ratio feature  float    -
    freight_ratio_missing feature    int    -
           total_weight_g feature  float    g
         total_volume_cm3 feature  float cm^3
  weight_missing_fraction feature  float    -
  volume_missing_fraction feature  float    -
         primary_category feature string    -
category_missing_fraction feature  float    -
           customer_state feature string    -
     primary_seller_state feature string    -
          distance_km_max feature  float   km
distance_missing_fraction feature  float    -
         interstate_share fea

In [9]:
# Missing counts and percentages, separately per population.
def miss_table(df, name):
    rows = []
    for c in ALLOW:
        n = int(df[c].isna().sum())
        rows.append({"column": c, "n_missing": n, "pct_missing": 100.0 * n / len(df)})
    t = pd.DataFrame(rows)
    return t.sort_values("n_missing", ascending=False)

for name, df in [("regression", reg), ("classification", cls)]:
    print(f"=== Missingness — {name} development population (n={len(df)}) ===")
    t = miss_table(df, name)
    print(t[t["n_missing"] > 0].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"(all other {len(t) - int((t['n_missing'] > 0).sum())} columns: 0 missing)")
    print()

=== Missingness — regression development population (n=77139) ===
                  column  n_missing  pct_missing
         distance_km_max        888        1.151
          total_weight_g         21        0.027
        total_volume_cm3         16        0.021
payment_installments_max          2        0.003
(all other 23 columns: 0 missing)

=== Missingness — classification development population (n=78938) ===
                   column  n_missing  pct_missing
          distance_km_max       1511        1.914
           total_weight_g        625        0.792
         total_volume_cm3        620        0.785
            freight_ratio        604        0.765
              total_price        604        0.765
         interstate_share        604        0.765
distance_missing_fraction        604        0.765
  volume_missing_fraction        604        0.765
category_missing_fraction        604        0.765
  weight_missing_fraction        604        0.765
            total_freight        6

In [10]:
# Numeric describe + constant-column flag, per population.
def numeric_describe(df, name):
    d = df[NUMS].describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]).T
    d["n_missing"] = [int(df[c].isna().sum()) for c in NUMS]
    d["constant"] = [int(df[c].nunique(dropna=True) <= 1) for c in NUMS]
    d = d[["count", "n_missing", "mean", "std", "min", "1%", "5%", "10%",
           "25%", "50%", "75%", "90%", "95%", "99%", "max", "constant"]]
    return d

for name, df in [("regression", reg), ("classification", cls)]:
    print(f"=== Numeric describe — {name} development population (n={len(df)}) ===")
    print(numeric_describe(df, name).to_string(float_format=lambda v: f"{v:.3f}"))
    print()

=== Numeric describe — regression development population (n=77139) ===


                              count  n_missing      mean       std     min      1%      5%      10%      25%      50%       75%       90%       95%        99%         max  constant
n_items                   77139.000          0     1.144     0.541   1.000   1.000   1.000    1.000    1.000    1.000     1.000     2.000     2.000      3.000      20.000         0
has_items                 77139.000          0     1.000     0.000   1.000   1.000   1.000    1.000    1.000    1.000     1.000     1.000     1.000      1.000       1.000         1
n_products                77139.000          0     1.039     0.227   1.000   1.000   1.000    1.000    1.000    1.000     1.000     1.000     1.000      2.000       7.000         0
n_sellers                 77139.000          0     1.014     0.125   1.000   1.000   1.000    1.000    1.000    1.000     1.000     1.000     1.000      2.000       5.000         0
n_categories              77139.000          0     0.994     0.149   0.000   0.000   1.000    1

In [11]:
# Category frequencies + class counts/proportions, per population.
def freq(df, col, top=8):
    vc = df[col].value_counts(dropna=False)
    t = pd.DataFrame({"count": vc, "proportion": vc / len(df)})
    return t.head(top)

for name, df in [("regression", reg), ("classification", cls)]:
    print(f"=== Category frequencies — {name} development population (n={len(df)}) ===")
    for col in ["primary_category", "primary_payment_type", "customer_state",
                "primary_seller_state", "purchase_month", "purchase_dayofweek",
                "purchase_hour"]:
        print(f"--- {col} ---")
        print(freq(df, col).to_string(float_format=lambda v: f"{v:.4f}"))
    print()

print("=== Class proportions (classification development population) ===")
vc = cls["is_detractor"].value_counts(dropna=False)
print(pd.DataFrame({"count": vc, "proportion": vc / len(cls)}).to_string(
    float_format=lambda v: f"{v:.4f}"))
print()
print("regression target lead_days is continuous (see numeric describe); no classes.")
print("classification target review_score_min distribution:")
vc = cls["review_score_min"].value_counts(dropna=False).sort_index()
print(pd.DataFrame({"count": vc, "proportion": vc / len(cls)}).to_string(
    float_format=lambda v: f"{v:.4f}"))

=== Category frequencies — regression development population (n=77139) ===
--- primary_category ---
                       count  proportion
primary_category                        
bed_bath_table          7272      0.0943
health_beauty           6875      0.0891
sports_leisure          6016      0.0780
computers_accessories   5225      0.0677
furniture_decor         4898      0.0635
housewares              4507      0.0584
watches_gifts           4437      0.0575
telephony               3236      0.0420
--- primary_payment_type ---
                      count  proportion
primary_payment_type                   
credit_card           58236      0.7549
boleto                15317      0.1986
voucher                2383      0.0309
debit_card             1202      0.0156
Unknown                   1      0.0000
--- customer_state ---
                count  proportion
customer_state                   
SP              32383      0.4198
RJ               9847      0.1277
MG               9078 

                  count  proportion
review_score_min                   
1.0                9113      0.1154
2.0                2513      0.0318
3.0                6520      0.0826
4.0               15265      0.1934
5.0               45527      0.5767


In [12]:
# Distributions of key price/weight/volume/distance/duration variables.
KEY = ["total_price", "total_freight", "total_weight_g", "total_volume_cm3",
       "distance_km_max", "lead_days"]

def key_dist(df, name):
    d = df[KEY].describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]).T
    d["skew"] = [float(df[c].skew()) for c in KEY]
    d["n_missing"] = [int(df[c].isna().sum()) for c in KEY]
    return d[["count", "n_missing", "mean", "std", "min", "1%", "5%", "10%",
              "25%", "50%", "75%", "90%", "95%", "99%", "max", "skew"]]

for name, df in [("regression", reg), ("classification", cls)]:
    print(f"=== Key-variable distributions — {name} development population (n={len(df)}) ===")
    print(key_dist(df, name).to_string(float_format=lambda v: f"{v:.3f}"))
    print("(lead_days is null for non-delivered orders; those appear only in the "
          "classification population.)")
    print()

=== Key-variable distributions — regression development population (n=77139) ===
                     count  n_missing      mean       std     min      1%      5%      10%      25%      50%       75%       90%       95%        99%         max   skew
total_price      77139.000          0   137.894   212.942   0.850  11.990  19.000   25.540   45.900   86.900   149.900   269.900   399.930    999.900   13440.000 10.240
total_freight    77139.000          0    22.826    20.772   0.000   7.390   7.880    9.340   13.870   17.190    24.055    39.560    55.140    105.165     711.330  6.538
total_weight_g   77118.000         21  2385.402  4756.313   2.000 100.000 150.000  175.000  300.000  750.000  2050.000  6550.000 10500.000  22350.000  184400.000  6.293
total_volume_cm3 77123.000         16 17338.581 29852.431 288.000 352.000 816.000 1408.000 2964.000 7200.000 19800.000 43320.000 64000.000 136000.000 1076544.000  6.300
distance_km_max  76251.000        888   602.527   592.677   0.000   5.863 

In [13]:
# Numeric correlations + selected feature/target relationships (development only).
def corr_target(df, target, name):
    s = df[NUMS].apply(lambda c: c.corr(df[target]))
    return s.abs().sort_values(ascending=False)

def top_pairwise(df, top=12):
    cm = df[NUMS].corr()
    pairs = []
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            pairs.append((NUMS[i], NUMS[j], abs(float(cm.iloc[i, j]))))
    pairs.sort(key=lambda t: -t[2])
    return pd.DataFrame(pairs[:top], columns=["a", "b", "abs_corr"])

print("=== Feature/target relationships — regression (lead_days) ===")
print(corr_target(reg, "lead_days", "regression").to_string(float_format=lambda v: f"{v:.3f}"))
print()
print("=== Feature/target relationships — classification (is_detractor) ===")
print(corr_target(cls, "is_detractor", "classification").to_string(float_format=lambda v: f"{v:.3f}"))
print()
print("=== Strongest pairwise numeric correlations (regression development) ===")
print(top_pairwise(reg).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("=== Strongest pairwise numeric correlations (classification development) ===")
print(top_pairwise(cls).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

=== Feature/target relationships — regression (lead_days) ===
distance_km_max             0.396
interstate_share            0.364
total_freight               0.177
freight_ratio               0.083
total_weight_g              0.075
total_volume_cm3            0.065
total_price                 0.054
payment_installments_max    0.054
n_sellers                   0.040
n_products                  0.031
n_items                     0.020
n_categories                0.020
payment_missing             0.016
distance_missing_fraction   0.005
category_missing_fraction   0.003
weight_missing_fraction     0.002
n_payment_methods           0.001
volume_missing_fraction     0.001
has_items                     NaN
freight_ratio_missing         NaN

=== Feature/target relationships — classification (is_detractor) ===


has_items                   0.155
freight_ratio_missing       0.155
total_freight               0.093
n_items                     0.087
total_volume_cm3            0.060
total_weight_g              0.055
n_categories                0.054
total_price                 0.048
interstate_share            0.048
distance_km_max             0.046
payment_installments_max    0.039
n_products                  0.034
category_missing_fraction   0.021
freight_ratio               0.012
payment_missing             0.009
distance_missing_fraction   0.005
n_payment_methods           0.003
n_sellers                   0.002
volume_missing_fraction     0.002
weight_missing_fraction     0.000

=== Strongest pairwise numeric correlations (regression development) ===


         a                b  abs_corr
   n_items        has_items       NaN
n_products        n_sellers     0.605
   n_items       n_products     0.465
   n_items    total_freight     0.453
   n_items        n_sellers     0.290
n_products     n_categories     0.285
   n_items total_volume_cm3     0.284
   n_items   total_weight_g     0.228
n_products    total_freight     0.195
   n_items      total_price     0.158
   n_items     n_categories     0.138
n_products total_volume_cm3     0.096

=== Strongest pairwise numeric correlations (classification development) ===


C:\Users\23745\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\23745\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


         a                     b  abs_corr
 has_items freight_ratio_missing     1.000
n_products             n_sellers     0.675
 has_items             n_sellers     0.581
 has_items          n_categories     0.503
   n_items            n_products     0.495
   n_items         total_freight     0.446
n_products          n_categories     0.415
n_products freight_ratio_missing     0.375
 has_items            n_products     0.375
   n_items             n_sellers     0.340
   n_items      total_volume_cm3     0.280
   n_items        total_weight_g     0.227


### Written observations (evidence + proposed experiments)

These are read-only findings from the tables above; nothing is deleted, imputed
or transformed here. Exact values are in the printed tables.

1. **Strong right-skew in money/weight/volume/distance.** `total_price`,
   `total_freight`, `total_weight_g`, `total_volume_cm3` and `distance_km_max`
   show `p50 ≪ mean` and `p99 ≫ p75` (large positive `skew`). Proposed
   experiment (later training phase, as a named variant): log / Box–Cox
   transform of strictly-positive money/weight/volume features for
   linear/logistic models; trees are scale-invariant and need no transform.

2. **Deterministic missingness-indicator redundancy.** By construction
   `has_items == int(n_items > 0)`, `freight_ratio_missing ==
   int(freight_ratio is null)` and `payment_missing == int(no usable payment)`.
   These are intentional missingness indicators retained per §11.3 ("retain all
   27 input columns"); they are flagged, not removed.

3. **Item-count multicollinearity.** `n_products`, `n_sellers` and
   `n_categories` grow with `n_items` (and `total_freight` with `total_price`),
   so the strongest pairwise numeric correlations are among these counts.
   Expected; trees absorb it, while linear models would need care with
   coefficient interpretation — do not compare coefficients across folds by
   position (§11.3).

4. **Geography coverage is uneven by state.** `distance_km_max` (and its
   `distance_missing_fraction`) is missing for a small share of orders
   (~2% full-population), but the full-population quality report shows it
   concentrates in specific states — DF ~9%, SC ~2–5%, RO ~2.5–4.8%. This
   reflects ZIP/state keys with no in-polygon surviving coordinate or an
   out-of-country median, not random missingness. Proposed experiment: a
   separately versioned historical-reference lookup for those states.

5. **Item-dimension and category missingness propagate.** `total_weight_g` /
   `total_volume_cm3` are null where any item dimension is invalid (→
   `weight_missing_fraction` / `volume_missing_fraction` > 0), and
   `category_missing_fraction` > 0 where items lack a translated category.
   `total_price` / `total_freight` are null for the small no-items cohort.
   These nulls are legitimate pipeline nulls left for fold-fitted imputation.

6. **Class imbalance (classification).** `is_detractor` ≈ 15% vs ≈ 85% in the
   development population (match the full-population 14,533 / 98,673 ≈ 14.7%).
   Stratified CV already preserves both classes per fold. Proposed: report
   precision/recall/PR-AUC and a threshold decision on the validation fold,
   not accuracy alone; only rebalance sampling with evidence from a held-out
   baseline.

7. **Calendar codes are categorical, not ordinal.** `purchase_month` (1–12),
   `purchase_dayofweek` (0–6) and `purchase_hour` (0–23) are nominal codes and
   must be one-hot encoded (§11.3), never treated as a single numeric scale.

No feature is dropped on the basis of these observations — any approved removal
would be a named variant with its own rationale and re-run.

## 6. Reproducibility

- The data files (`orders_ml_features.csv`, `zip_centroids.csv`,
  `split_assignments.csv`, `cv_assignments.csv`) are byte-identical across runs
  (no timestamps in the data). The JSON reports carry `run_timestamp` /
  `retrieved_at` metadata fields, so only those fields differ between runs.
- Deterministic seeds: outer/inner CV use `random_state=42`; primary item/payment
  tie-breaks and all sort orders use `kind="mergesort"`.
- The geography `_point_covers` helper is exactly boundary-inclusive: a fast
  `contains_xy` interior pass plus an exact `covers` re-check on the tiny
  non-interior set (full geolocation table: 738,332 rows → 738,303 accepted,
  29 outside the polygon).
- To run without the notebook:

  ```
  python -m src.features.build_features
  python -m src.features.create_splits
  ```

- Training-fitted imputation/scaling/encoding, seller outcome histories and
  `T_pred` are intentionally **not** implemented this round; the §9 checks that
  depend on them are recorded under `quality_report.json → deferred_checks`.